# Exploration des données et des labels (EDA)

**Projet** : PrediCare TPA
**Objectif** : explorer les 12 CSV métiers et les 2 fichiers de labels : jointures, taux de fraude, fuite de données, qualité.
**Références** : Dictionnaire des données §4 à §6, backlog PB-03.

**Règles de lecture**
- Tous les chiffres viennent de cellules exécutées. Les conclusions et gravités sont **générées à partir des résultats** (sections 7 et 8).
- `identites_pii.csv` : seul `assure_id` est chargé (comptage). Aucune donnée personnelle n'est affichée.
- Les sorties sont agrégées : aucun dump de lignes.
- Un signal ou un contrôle dont les colonnes sont introuvables est marqué **NON ÉVALUÉ** avec la raison. Il n'est jamais ignoré en silence.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 80)
np.random.seed(42)

DATA_DIR = next((p for p in (Path("../data/raw"), Path("data/raw")) if p.exists()), None)
assert DATA_DIR is not None, "data/raw introuvable : STOP, vérifier l'emplacement des données."
print("Données lues dans :", DATA_DIR.resolve())


def find_file(name):
    hits = sorted(DATA_DIR.rglob(name))
    assert hits, f"Fichier manquant sous {DATA_DIR} : {name}. STOP : ne pas inventer de données."
    return hits[0]


def col(df, *cands, contains=None):
    """Première colonne existante parmi cands, sinon première dont le nom contient tous les mots de contains."""
    for c in cands:
        if c in df.columns:
            return c
    if contains:
        for c in df.columns:
            if all(k in c.lower() for k in contains):
                return c
    return None


def need(df, label, *cands, contains=None):
    c = col(df, *cands, contains=contains)
    if c is None:
        raise KeyError(
            f"Colonne introuvable pour « {label} ». Colonnes disponibles : {list(df.columns)}"
        )
    return c

Données lues dans : /mnt/c/Users/Asus/Documents/Formation ODC & IngeData/Capstone/predicare-tpa/PrediCare/data/raw


## 1. Volumétrie, schéma réel et profil des valeurs manquantes

In [ ]:
FILES = [
    "assures.csv",
    "contrats.csv",
    "factures.csv",
    "garanties.csv",
    "grille_tarifaire.csv",
    "identites_pii.csv",
    "lignes_facture.csv",
    "ordonnance_lignes.csv",
    "ordonnances.csv",
    "prestataires.csv",
    "referentiel_actes.csv",
    "referentiel_medicaments.csv",
    "labels_fraude.csv",
    "labels_prestataires.csv",
]
PII = "identites_pii.csv"
dfs, date_coerce, pii_info, vol = {}, {}, {}, []

for name in FILES:
    p = find_file(name)
    header = pd.read_csv(p, nrows=0).columns.tolist()
    taille = round(p.stat().st_size / 1048576, 2)
    if name == PII:
        assert "assure_id" in header, "assure_id absent de identites_pii.csv"
        ids = pd.read_csv(p, usecols=["assure_id"], dtype=str)
        pii_info = {"lignes": len(ids), "colonnes": len(header)}
        dfs[name] = ids
        vol.append(
            {
                "fichier": name,
                "lignes": len(ids),
                "colonnes": len(header),
                "taille_mo": taille,
                "note": "assure_id seul chargé",
            }
        )
        continue
    dtype = {
        c: str
        for c in header
        if c.endswith("_id") or c.startswith("code") or c.startswith("numero")
    }
    d = pd.read_csv(p, dtype=dtype, low_memory=False)
    coerce = {}
    for c in header:
        if c.startswith("date_") or c == "date":
            parsed = pd.to_datetime(d[c], errors="coerce", format="%Y-%m-%d")
            coerce[c] = int((parsed.isna() & d[c].notna()).sum())
            d[c] = parsed
    date_coerce[name] = coerce
    dfs[name] = d
    vol.append(
        {
            "fichier": name,
            "lignes": len(d),
            "colonnes": d.shape[1],
            "taille_mo": taille,
            "note": str(p.parent.name),
        }
    )

print("=== VOLUMÉTRIE ===")
print(pd.DataFrame(vol).to_string(index=False))

=== VOLUMÉTRIE ===
                    fichier  lignes  colonnes  taille_mo                  note
                assures.csv    4459         9       0.36                   raw
               contrats.csv    1500        12       0.18                   raw
               factures.csv   84593        24      16.29                   raw
              garanties.csv      27        11       0.01                   raw
       grille_tarifaire.csv    3242         3       0.08                   raw
          identites_pii.csv    4459        10       0.54 assure_id seul chargé
         lignes_facture.csv  123270         9      11.50                   raw
      ordonnance_lignes.csv   82175         6       7.52                   raw
            ordonnances.csv   47804         8       5.11                   raw
           prestataires.csv     251        20       0.05                   raw
      referentiel_actes.csv      58        12       0.00                   raw
referentiel_medicaments.csv      

In [ ]:
print("=== SCHÉMA RÉEL (colonne:type) ===")
for name, d in dfs.items():
    if name == PII:
        print(
            f"\n{name}: {pii_info['colonnes']} colonnes, contenu non chargé (données personnelles)"
        )
        continue
    print(f"\n{name} ({len(d):,} lignes)")
    print("  " + ", ".join(f"{c}:{t}" for c, t in d.dtypes.astype(str).items()))

print("\n=== COLONNES AVEC VALEURS MANQUANTES ===")
miss_rows = []
for name, d in dfs.items():
    if name == PII:
        continue
    m = d.isna().mean() * 100
    for c, v in m[m > 0].items():
        miss_rows.append(
            {
                "fichier": name,
                "colonne": c,
                "pct_manquant": round(v, 2),
                "nb": int(d[c].isna().sum()),
            }
        )
print(pd.DataFrame(miss_rows).to_string(index=False) if miss_rows else "Aucune valeur manquante.")

fac0 = dfs["factures.csv"]
print("\n=== NULLS STRUCTURELS : lien avec le statut ===")
if col(fac0, "statut") and col(fac0, "motif_rejet"):
    print(pd.crosstab(fac0["statut"], fac0["motif_rejet"].isna().rename("motif_rejet_nul")))
if col(fac0, "statut") and col(fac0, "date_traitement"):
    print(
        pd.crosstab(fac0["statut"], fac0["date_traitement"].isna().rename("date_traitement_nulle"))
    )
if col(fac0, "statut") and col(fac0, "montant_pris_en_charge"):
    print(pd.crosstab(fac0["statut"], fac0["montant_pris_en_charge"].isna().rename("pec_nul")))

=== SCHÉMA RÉEL (colonne:type) ===

assures.csv (4,459 lignes)
  assure_id:str, contrat_id:str, rang_beneficiaire:str, assure_principal_id:str, sexe:str, annee_naissance:int64, ville:str, date_adhesion:datetime64[us], date_radiation:datetime64[us]

contrats.csv (1,500 lignes)
  contrat_id:str, numero_police:str, souscripteur_id:str, souscripteur_nom:str, secteur:str, formule:str, date_effet_police:datetime64[us], date_effet:datetime64[us], date_fin:datetime64[us], statut:str, nb_beneficiaires:int64, prime_annuelle_totale:int64

factures.csv (84,593 lignes)
  facture_id:str, numero_facture:str, prestataire_id:str, assure_id:str, contrat_id:str, ordonnance_id:str, type_facture:str, date_soin:datetime64[us], heure_soin:str, date_facture:datetime64[us], date_reception:datetime64[us], canal_reception:str, mode_reglement:str, attestation_pec_id:str, entente_prealable_id:str, devise:str, nb_lignes:int64, montant_total:int64, montant_eligible:int64, montant_pris_en_charge:int64, reste_a_charge

## 2. Clés, intégrité référentielle et cardinalités

In [ ]:
PK_SPEC = {
    "assures.csv": [("assure_id",)],
    "contrats.csv": [("contrat_id",)],
    "factures.csv": [("facture_id",)],
    "lignes_facture.csv": [("ligne_id",)],
    "ordonnances.csv": [("ordonnance_id",)],
    "ordonnance_lignes.csv": [("ordonnance_id",), ("num_ligne", "ligne_id", "ligne", "rang")],
    "prestataires.csv": [("prestataire_id",)],
    "referentiel_actes.csv": [("code_acte", "code")],
    "referentiel_medicaments.csv": [("code_medicament", "code")],
    "grille_tarifaire.csv": [("prestataire_id",), ("code", "code_acte")],
    "garanties.csv": [("formule",), ("categorie",)],
    "labels_fraude.csv": [("facture_id",)],
    "labels_prestataires.csv": [("prestataire_id",)],
    "identites_pii.csv": [("assure_id",)],
}
rows = []
for name, spec in PK_SPEC.items():
    d = dfs[name]
    cols = [col(d, *c) for c in spec]
    if None in cols:
        rows.append(
            {"fichier": name, "cle": str(spec), "doublons": "colonne introuvable", "nulls_cle": ""}
        )
        continue
    rows.append(
        {
            "fichier": name,
            "cle": " + ".join(cols),
            "doublons": int(d.duplicated(subset=cols).sum()),
            "nulls_cle": int(d[cols].isna().any(axis=1).sum()),
        }
    )
print("=== 2.1 UNICITÉ DES CLÉS ===")
print(pd.DataFrame(rows).to_string(index=False))

FK_SPEC = [
    ("assures.csv", ("contrat_id",), "contrats.csv", ("contrat_id",)),
    ("factures.csv", ("assure_id",), "assures.csv", ("assure_id",)),
    ("factures.csv", ("prestataire_id",), "prestataires.csv", ("prestataire_id",)),
    ("factures.csv", ("contrat_id",), "contrats.csv", ("contrat_id",)),
    ("factures.csv", ("ordonnance_id",), "ordonnances.csv", ("ordonnance_id",)),
    ("lignes_facture.csv", ("facture_id",), "factures.csv", ("facture_id",)),
    ("ordonnances.csv", ("assure_id",), "assures.csv", ("assure_id",)),
    (
        "ordonnances.csv",
        ("prescripteur_id", "prestataire_id", "prescripteur"),
        "prestataires.csv",
        ("prestataire_id",),
    ),
    ("ordonnance_lignes.csv", ("ordonnance_id",), "ordonnances.csv", ("ordonnance_id",)),
    ("grille_tarifaire.csv", ("prestataire_id",), "prestataires.csv", ("prestataire_id",)),
    ("identites_pii.csv", ("assure_id",), "assures.csv", ("assure_id",)),
    ("labels_fraude.csv", ("facture_id",), "factures.csv", ("facture_id",)),
    ("labels_prestataires.csv", ("prestataire_id",), "prestataires.csv", ("prestataire_id",)),
]
rows, orph_total = [], 0
for f_src, c_src, f_ref, c_ref in FK_SPEC:
    cs, cr = col(dfs[f_src], *c_src), col(dfs[f_ref], *c_ref)
    if cs is None or cr is None:
        rows.append(
            {
                "relation": f"{f_src} -> {f_ref}",
                "non_nuls": "",
                "orphelins": "colonne introuvable",
                "pct": "",
            }
        )
        continue
    s = dfs[f_src][cs].dropna()
    n_orph = int((~s.isin(set(dfs[f_ref][cr].dropna()))).sum())
    orph_total += n_orph
    rows.append(
        {
            "relation": f"{f_src}.{cs} -> {f_ref}.{cr}",
            "non_nuls": len(s),
            "orphelins": n_orph,
            "pct": round(n_orph / len(s) * 100, 3) if len(s) else 0,
        }
    )
print("\n=== 2.2 ORPHELINS ===")
print(pd.DataFrame(rows).to_string(index=False))

fac0, lig0 = dfs["factures.csv"], dfs["lignes_facture.csv"]
act0, med0 = dfs["referentiel_actes.csv"], dfs["referentiel_medicaments.csv"]
c_lcode0 = need(lig0, "code (lignes_facture)", "code", "code_acte")
c_ltype0 = need(lig0, "type de ligne", "type_ligne", contains=["type"])
codes_act = set(act0[need(act0, "code acte", "code_acte", "code")])
codes_med = set(med0[need(med0, "code médicament", "code_medicament", "code")])
print("\n=== 2.3 CODES DES LIGNES VS RÉFÉRENTIELS ===")
print("Modalités type_ligne :", lig0[c_ltype0].value_counts(dropna=False).to_dict())
m_a, m_m = lig0[c_ltype0] == "ACTE", lig0[c_ltype0] == "MEDICAMENT"
orph_codes_act = int((~lig0.loc[m_a, c_lcode0].isin(codes_act)).sum())
orph_codes_med = int((~lig0.loc[m_m, c_lcode0].isin(codes_med)).sum())
print(f"Lignes ACTE avec code absent du référentiel : {orph_codes_act:,} / {int(m_a.sum()):,}")
print(
    f"Lignes MEDICAMENT avec code absent du référentiel : {orph_codes_med:,} / {int(m_m.sum()):,}"
)

print("\n=== 2.4 CARDINALITÉS ===")
n_l = lig0.groupby("facture_id").size()
print("Lignes par facture :", n_l.describe().round(2).to_dict())
print("Factures sans ligne :", len(set(fac0["facture_id"]) - set(n_l.index)))
ordo0, ol0 = dfs["ordonnances.csv"], dfs["ordonnance_lignes.csv"]
print("Ordonnances sans ligne :", len(set(ordo0["ordonnance_id"]) - set(ol0["ordonnance_id"])))
c_ord0 = need(fac0, "ordonnance (factures)", "ordonnance_id")
print(
    f"Factures sans ordonnance_id : {int(fac0[c_ord0].isna().sum()):,} ({fac0[c_ord0].isna().mean():.1%})"
)
print("Nombre de factures par ordonnance (valeur : nb d'ordonnances) :")
print(fac0.dropna(subset=[c_ord0]).groupby(c_ord0).size().value_counts().sort_index().to_dict())
mm = (
    fac0[["facture_id", "assure_id", c_ord0]]
    .dropna(subset=[c_ord0])
    .merge(
        ordo0[["ordonnance_id", "assure_id"]].rename(columns={"assure_id": "_ass_o"}),
        left_on=c_ord0,
        right_on="ordonnance_id",
        how="left",
    )
)
n_mis = int((mm["_ass_o"].notna() & (mm["_ass_o"] != mm["assure_id"])).sum())
print(f"Factures dont l'ordonnance appartient à un autre assuré : {n_mis:,}")

=== 2.1 UNICITÉ DES CLÉS ===
                    fichier                       cle  doublons  nulls_cle
                assures.csv                 assure_id         0          0
               contrats.csv                contrat_id         0          0
               factures.csv                facture_id         0          0
         lignes_facture.csv                  ligne_id         0          0
            ordonnances.csv             ordonnance_id         0          0
      ordonnance_lignes.csv ordonnance_id + num_ligne         0          0
           prestataires.csv            prestataire_id         0          0
      referentiel_actes.csv                 code_acte         0          0
referentiel_medicaments.csv           code_medicament         0          0
       grille_tarifaire.csv     prestataire_id + code         0          0
              garanties.csv       formule + categorie         0          0
          labels_fraude.csv                facture_id         0        

Lignes par facture : {'count': 84593.0, 'mean': 1.46, 'std': 0.66, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 6.0}
Factures sans ligne : 0
Ordonnances sans ligne : 0
Factures sans ordonnance_id : 37,688 (44.6%)
Nombre de factures par ordonnance (valeur : nb d'ordonnances) :
{1: 44501, 2: 824, 3: 44, 4: 14, 5: 25, 6: 36, 7: 24, 8: 4, 9: 3}
Factures dont l'ordonnance appartient à un autre assuré : 175


## 3. Labels et taux de fraude (Dictionnaire §5 et §6)

In [ ]:
fac = dfs["factures.csv"]
lab, plab = dfs["labels_fraude.csv"], dfs["labels_prestataires.csv"]
TYPES14 = [
    "SURFACTURATION",
    "QUANTITE_EXCESSIVE",
    "UPCODING",
    "LIGNES_NON_PRESCRITES",
    "SUBSTITUTION_PRODUIT",
    "ORDONNANCE_EXPIREE",
    "SOINS_JOUR_FERME",
    "DOUBLON",
    "FRACTIONNEMENT",
    "SURCONSOMMATION",
    "ACTE_FICTIF",
    "INCOHERENCE_PROFIL",
    "PRESTATAIRE_RADIE",
    "HORS_COUVERTURE",
]
print(f"Factures sans label : {int((~fac['facture_id'].isin(lab['facture_id'])).sum()):,}")
print(f"Labels sans facture : {int((~lab['facture_id'].isin(fac['facture_id'])).sum()):,}")

df = fac.merge(lab, on="facture_id", how="left", validate="one_to_one").merge(
    plab, on="prestataire_id", how="left", validate="many_to_one"
)
df["est_fraude"] = pd.to_numeric(df["est_fraude"], errors="coerce")
lab_ok = df[df["est_fraude"].notna()].copy()
N, N_F = len(lab_ok), int(lab_ok["est_fraude"].sum())
TAUX = N_F / N
print(f"\nFactures étiquetées : {N:,} | fraudes : {N_F:,} | taux global : {TAUX:.2%}")

print("\n=== 3.1 PAR TYPE DE FRAUDE ===")
tf = lab_ok[lab_ok["est_fraude"] == 1]["type_fraude"].value_counts(dropna=False)
print(
    pd.DataFrame(
        {
            "type_fraude": tf.index,
            "volume": tf.values,
            "pct_des_fraudes": (tf.values / N_F * 100).round(2),
            "pct_des_factures": (tf.values / N * 100).round(3),
        }
    ).to_string(index=False)
)
present = set(tf.index.dropna())
missing_types, extra_types = sorted(set(TYPES14) - present), sorted(present - set(TYPES14))
types_ok = not missing_types and not extra_types
print(
    f"Types présents : {len(present)} / 14 attendus | manquants : {missing_types or 'aucun'} | inattendus : {extra_types or 'aucun'}"
)
print(
    "type_fraude sur les factures non frauduleuses :",
    lab_ok[lab_ok["est_fraude"] == 0]["type_fraude"].value_counts(dropna=False).head(5).to_dict(),
)

print("\n=== 3.2 PAR TYPE DE FACTURE ===")
tcol = col(lab_ok, "type_facture", "type")
if tcol:
    t = lab_ok.groupby(tcol)["est_fraude"].agg(factures="size", fraudes="sum", taux_pct="mean")
    t["taux_pct"] = (t["taux_pct"] * 100).round(2)
    print(t)
else:
    print("NON ÉVALUÉ : colonne type de facture introuvable. Colonnes :", list(lab_ok.columns))

print("\n=== 3.3 PAR PRESTATAIRE ===")
gp = (
    lab_ok.groupby("prestataire_id")["est_fraude"]
    .agg(n="size", f="sum", taux="mean")
    .sort_values("f", ascending=False)
)
k = max(1, int(len(gp) * 0.1))
print(f"Prestataires : {len(gp):,} dont {int((gp['f'] > 0).sum()):,} avec au moins une fraude")
print(
    f"Part des fraudes portée par les 10 % de prestataires les plus fraudeurs : {gp['f'].head(k).sum() / N_F:.1%}"
)
print(
    "Distribution du taux de fraude par prestataire :",
    (gp["taux"] * 100).describe().round(2).to_dict(),
)
if "est_suspect" in lab_ok:
    s = lab_ok.groupby(lab_ok["est_suspect"].fillna(-1))["est_fraude"].agg(
        factures="size", fraudes="sum", taux_pct="mean"
    )
    s["taux_pct"] = (s["taux_pct"] * 100).round(2)
    s["part_des_fraudes_pct"] = (s["fraudes"] / N_F * 100).round(1)
    print("\nPar est_suspect (-1 = prestataire sans label) :")
    print(s)

print("\n=== 3.4 DANS LE TEMPS (date_soin) ===")
c_ds = need(lab_ok, "date de soin", "date_soin")
par_mois = (
    lab_ok.assign(mois=lab_ok[c_ds].dt.to_period("M").astype(str))
    .groupby("mois")["est_fraude"]
    .agg(factures="size", fraudes="sum", taux_pct="mean")
)
par_mois["taux_pct"] = (par_mois["taux_pct"] * 100).round(2)
print(par_mois)

print("\n=== 3.5 est_valide ET motif_invalidite ===")
if "est_valide" in lab_ok:
    print(pd.crosstab(lab_ok["est_valide"], lab_ok["est_fraude"], margins=True))
if "motif_invalidite" in lab_ok:
    print("\nmotif_invalidite x est_fraude :")
    print(pd.crosstab(lab_ok["motif_invalidite"].fillna("(aucun)"), lab_ok["est_fraude"]))
print(
    "Interprétation : à lire dans les tableaux ci-dessus, le dictionnaire ne définit pas est_valide."
)

Factures sans label : 0
Labels sans facture : 0



Factures étiquetées : 84,593 | fraudes : 7,347 | taux global : 8.69%

=== 3.1 PAR TYPE DE FRAUDE ===
          type_fraude  volume  pct_des_fraudes  pct_des_factures
       SURFACTURATION    1272            17.31             1.504
              DOUBLON     909            12.37             1.075
             UPCODING     742            10.10             0.877
LIGNES_NON_PRESCRITES     733             9.98             0.867
   QUANTITE_EXCESSIVE     568             7.73             0.671
   ORDONNANCE_EXPIREE     502             6.83             0.593
          ACTE_FICTIF     496             6.75             0.586
      SURCONSOMMATION     469             6.38             0.554
   INCOHERENCE_PROFIL     379             5.16             0.448
 SUBSTITUTION_PRODUIT     368             5.01             0.435
       FRACTIONNEMENT     340             4.63             0.402
     SOINS_JOUR_FERME     227             3.09             0.268
      HORS_COUVERTURE     195             2.65       

est_fraude            0     1
motif_invalidite             
(aucun)           77246  7347
Interprétation : à lire dans les tableaux ci-dessus, le dictionnaire ne définit pas est_valide.


## 4. Fuite de données (Dictionnaire §4)

Le dictionnaire indique que `statut`, `motif_rejet`, `montant_pris_en_charge` et `date_traitement` résultent de la décision historique et sont partiellement corrélées à la fraude : elles ne doivent pas servir de variables du modèle.
Cette section **mesure** cette dépendance. L'AUC univariée vaut 0,5 pour une variable sans lien avec la fraude. La colonne `force` est max(AUC, 1-AUC). Pour les variables catégorielles, l'encodage par taux de fraude est calculé sur tout le jeu : la mesure est donc optimiste, ce qui suffit pour détecter une fuite.

In [ ]:
EXCLUDED_FEATURES = ["statut", "motif_rejet", "montant_pris_en_charge", "date_traitement"]
print(
    "Colonnes de fuite absentes du fichier :",
    [c for c in EXCLUDED_FEATURES if c not in df.columns] or "aucune",
)


def auc_rank(score, y):
    d = pd.DataFrame({"s": score, "y": y}).dropna()
    n1, n0 = int((d["y"] == 1).sum()), int((d["y"] == 0).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = d["s"].rank(method="average")
    return float((r[d["y"] == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def auc_cat(series, y):
    d = pd.DataFrame({"c": series.astype("object").where(series.notna(), "__NA__"), "y": y}).dropna(
        subset=["y"]
    )
    return auc_rank(d.groupby("c")["y"].transform("mean"), d["y"])


lk = df[df["est_fraude"].notna()].copy()
yk = lk["est_fraude"]
rows = []

if "statut" in lk:
    print("\n=== statut vs fraude ===")
    t = lk.groupby("statut", dropna=False)["est_fraude"].agg(
        n="size", fraudes="sum", taux_pct="mean"
    )
    t["taux_pct"] = (t["taux_pct"] * 100).round(2)
    print(t)
    rows.append({"variable": "statut (modalités)", "auc": auc_cat(lk["statut"], yk)})

if "motif_rejet" in lk:
    print("\n=== motif_rejet vs fraude ===")
    t = (
        lk.assign(motif=lk["motif_rejet"].fillna("(aucun)"))
        .groupby("motif")["est_fraude"]
        .agg(n="size", fraudes="sum", taux_pct="mean")
    )
    t["taux_pct"] = (t["taux_pct"] * 100).round(2)
    print(t.sort_values("taux_pct", ascending=False))
    rows.append({"variable": "motif_rejet (modalités)", "auc": auc_cat(lk["motif_rejet"], yk)})

if "montant_pris_en_charge" in lk:
    print("\n=== montant_pris_en_charge vs fraude ===")
    pec = pd.to_numeric(lk["montant_pris_en_charge"], errors="coerce")
    print(pec.groupby(yk).agg(["count", "mean", "median"]).round(1))
    print("Part de nuls par classe (%) :", (pec.isna().groupby(yk).mean() * 100).round(2).to_dict())
    rows.append(
        {
            "variable": "montant_pris_en_charge (valeur, nul = -1)",
            "auc": auc_rank(pec.fillna(-1), yk),
        }
    )
    rows.append(
        {
            "variable": "montant_pris_en_charge (indicateur nul)",
            "auc": auc_rank(pec.isna().astype(int), yk),
        }
    )
    c_base = col(lk, "montant_demande", "montant_total", contains=["montant", "total"])
    if c_base:
        ratio_pec = pec / pd.to_numeric(lk[c_base], errors="coerce").replace(0, np.nan)
        rows.append(
            {"variable": f"pris_en_charge / {c_base}", "auc": auc_rank(ratio_pec.fillna(-1), yk)}
        )

if "date_traitement" in lk:
    print("\n=== date_traitement vs fraude ===")
    ref = col(lk, "date_reception", "date_facture", "date_soin")
    dt_null = lk["date_traitement"].isna()
    print("Part de nulles par classe (%) :", (dt_null.groupby(yk).mean() * 100).round(2).to_dict())
    rows.append(
        {"variable": "date_traitement (indicateur nul)", "auc": auc_rank(dt_null.astype(int), yk)}
    )
    if ref:
        delai = (lk["date_traitement"] - lk[ref]).dt.days
        print(f"Délai date_traitement - {ref} (jours) par classe :")
        print(delai.groupby(yk).describe().round(2))
        rows.append({"variable": f"délai date_traitement - {ref}", "auc": auc_rank(delai, yk)})

c_ref_m = col(lk, "montant_total", contains=["montant", "total"])
if c_ref_m:
    rows.append(
        {
            "variable": f"{c_ref_m} (référence : variable légitime)",
            "auc": auc_rank(pd.to_numeric(lk[c_ref_m], errors="coerce"), yk),
        }
    )

leak_df = pd.DataFrame(rows)
leak_df["auc"] = leak_df["auc"].round(3)
leak_df["force"] = leak_df["auc"].apply(
    lambda a: round(max(a, 1 - a), 3) if pd.notna(a) else np.nan
)
print("\n=== AUC UNIVARIÉE ===")
print(leak_df.to_string(index=False))
print(
    f"\nDÉCISION : les colonnes {EXCLUDED_FEATURES} sont exclues des features d'entraînement ET d'inférence."
)

Colonnes de fuite absentes du fichier : aucune

=== statut vs fraude ===
              n  fraudes  taux_pct
statut                            
EN_COURS   2826      249      8.81
PAYEE     79775     5709      7.16
REJETEE    1992     1389     69.73

=== motif_rejet vs fraude ===
                                 n  fraudes  taux_pct
motif                                                
CONTROLE_MEDICAL               998      998    100.00
PRODUIT_NON_REMBOURSABLE         3        3    100.00
HORS_PERIODE_COUVERTURE         69       69    100.00
ORDONNANCE_PERIMEE             246      246    100.00
ENTENTE_PREALABLE_ABSENTE       15        9     60.00
PLAFOND_ATTEINT                 51       17     33.33
DELAI_DE_CARENCE               151       20     13.25
(aucun)                      82601     5958      7.21
PIECE_MANQUANTE_ATTESTATION    459       27      5.88

=== montant_pris_en_charge vs fraude ===
            count     mean   median
est_fraude                         
0           7

Délai date_traitement - date_reception (jours) par classe :
              count  mean   std  min  25%  50%   75%   max
est_fraude                                                
0           74669.0  8.50  4.04  2.0  5.0  8.0  12.0  15.0
1            7098.0  8.44  4.04  2.0  5.0  8.0  12.0  15.0

=== AUC UNIVARIÉE ===
                                     variable   auc  force
                           statut (modalités) 0.594  0.594
                      motif_rejet (modalités) 0.592  0.592
    montant_pris_en_charge (valeur, nul = -1) 0.565  0.565
      montant_pris_en_charge (indicateur nul) 0.500  0.500
               pris_en_charge / montant_total 0.327  0.673
             date_traitement (indicateur nul) 0.500  0.500
       délai date_traitement - date_reception 0.496  0.504
montant_total (référence : variable légitime) 0.726  0.726

DÉCISION : les colonnes ['statut', 'motif_rejet', 'montant_pris_en_charge', 'date_traitement'] sont exclues des features d'entraînement ET d'inférenc

## 5. Signaux du §6 : vérification exploratoire

Pour chacun des 14 types, un signal est calculé à partir des données (jointures facture ↔ lignes ↔ ordonnance ↔ référentiels), puis croisé avec `est_fraude`.
- `precision` : part de fraudes parmi les factures signalées. `lift` : precision / taux global.
- `rappel_type_attendu` : part des fraudes du type visé que le signal capte.
- Les colonnes utilisées sont affichées ci-dessous pour contrôle. Un signal impossible à calculer apparaît comme NON ÉVALUÉ avec la raison.
- Ce sont des heuristiques d'exploration, pas des features finales. La carence (HORS_COUVERTURE) n'est pas évaluée ici.

In [ ]:
fac = df.copy()
fac_s = fac.set_index("facture_id")
y, typ = fac_s["est_fraude"], fac_s["type_fraude"]

C_ASS = need(fac, "assure (factures)", "assure_id")
C_PRE = need(fac, "prestataire (factures)", "prestataire_id")
C_ORD = need(fac, "ordonnance (factures)", "ordonnance_id")
C_CTR = need(fac, "contrat (factures)", "contrat_id")
C_DS = need(fac, "date de soin", "date_soin")
C_MT = need(fac, "montant total", "montant_total", contains=["montant", "total"])

lig = dfs["lignes_facture.csv"]
C_LCODE = need(lig, "code (lignes_facture)", "code", "code_acte")
C_LTYPE = need(lig, "type de ligne", "type_ligne", contains=["type"])
C_LQTE = need(lig, "quantité facturée", "quantite", contains=["quant"])
C_LPU = need(lig, "prix unitaire", "prix_unitaire", contains=["prix"])
extra = [c for c in (C_PRE, C_ASS, C_ORD, C_DS) if c not in lig.columns]
lf = lig.merge(fac[["facture_id"] + extra], on="facture_id", how="left")

act, med = dfs["referentiel_actes.csv"], dfs["referentiel_medicaments.csv"]
pre, ass, con = dfs["prestataires.csv"], dfs["assures.csv"], dfs["contrats.csv"]
ordo, ol, gr = dfs["ordonnances.csv"], dfs["ordonnance_lignes.csv"], dfs["grille_tarifaire.csv"]
A_CODE = need(act, "code (réf. actes)", "code_acte", "code")
M_CODE = need(med, "code (réf. médicaments)", "code_medicament", "code")

resolved = {}
results = {}


def by_fact(mask, ids):
    return pd.Series(np.asarray(mask, dtype=bool), index=ids.values).groupby(level=0).any()


def run_signal(name, expected, fn):
    try:
        s = fn()
        s = s[~s.index.duplicated()].reindex(fac_s.index).fillna(False).astype(bool)
        results[name] = (expected, s, None)
    except Exception as e:
        results[name] = (expected, None, f"{type(e).__name__}: {str(e)[:200]}")


def s_surf_acte():
    c_max = need(act, "tarif maximum", "tarif_max", "tarif_maximum", contains=["max"])
    resolved["1 tarif max (réf. actes)"] = c_max
    a = (
        act[[A_CODE, c_max]]
        .rename(columns={A_CODE: "_acode", c_max: "_tmax"})
        .drop_duplicates("_acode")
    )
    L = lf[lf[C_LTYPE] == "ACTE"].merge(a, left_on=C_LCODE, right_on="_acode", how="left")
    return by_fact(L[C_LPU] > L["_tmax"], L["facture_id"])


def s_surf_med():
    c_ref = need(med, "prix de référence", "prix_reference", "prix_ref", contains=["prix"])
    resolved["1b prix réf. (médicaments)"] = c_ref
    m = (
        med[[M_CODE, c_ref]]
        .rename(columns={M_CODE: "_mcode", c_ref: "_pref"})
        .drop_duplicates("_mcode")
    )
    L = lf[lf[C_LTYPE] == "MEDICAMENT"].merge(m, left_on=C_LCODE, right_on="_mcode", how="left")
    return by_fact(L[C_LPU] > L["_pref"], L["facture_id"])


def s_surf_grille(facteur):
    gp = need(gr, "prestataire (grille)", "prestataire_id")
    gc = need(gr, "code (grille)", "code", "code_acte")
    gt = need(gr, "tarif (grille)", "tarif", "tarif_pratique", contains=["tarif"])
    resolved["1c/1d tarif grille"] = gt
    g = (
        gr[[gp, gc, gt]]
        .rename(columns={gp: "_gp", gc: "_gc", gt: "_tg"})
        .drop_duplicates(["_gp", "_gc"])
    )
    L = lf.merge(g, left_on=[C_PRE, C_LCODE], right_on=["_gp", "_gc"], how="left")
    return by_fact(L[C_LPU] > L["_tg"] * facteur, L["facture_id"])


def s_qte():
    o_code = need(ol, "code (lignes ordonnance)", "code", "code_acte", contains=["code"])
    o_qte = need(ol, "quantité prescrite", "quantite", contains=["quant"])
    resolved["2 ordonnance_lignes code/qté"] = f"{o_code} / {o_qte}"
    P = (
        ol.groupby(["ordonnance_id", o_code], as_index=False)[o_qte]
        .sum()
        .rename(columns={o_code: "_c", o_qte: "_qp"})
    )
    L = lf[lf[C_ORD].notna()].merge(
        P, left_on=[C_ORD, C_LCODE], right_on=["ordonnance_id", "_c"], how="inner"
    )
    return by_fact(L[C_LQTE] > L["_qp"], L["facture_id"])


def _non_prescrites(only_med):
    o_code = need(ol, "code (lignes ordonnance)", "code", "code_acte", contains=["code"])
    keys = (
        ol[["ordonnance_id", o_code]].drop_duplicates().rename(columns={o_code: "_c"}).assign(_ok=1)
    )
    L = lf[lf[C_ORD].notna()]
    if only_med:
        L = L[L[C_LTYPE] == "MEDICAMENT"]
    L = L.merge(keys, left_on=[C_ORD, C_LCODE], right_on=["ordonnance_id", "_c"], how="left")
    return L, L["_ok"].isna()


def s_non_prescrites():
    L, np_ = _non_prescrites(False)
    return by_fact(np_, L["facture_id"])


def s_substitution():
    c_ref = need(med, "prix de référence", "prix_reference", "prix_ref", contains=["prix"])
    thr = pd.to_numeric(med[c_ref], errors="coerce").quantile(0.75)
    resolved["5 seuil prix médicament (P75 réf.)"] = round(float(thr), 1)
    L, np_ = _non_prescrites(True)
    return by_fact(np_ & (L[C_LPU] > thr), L["facture_id"])


def s_expiree():
    c_od = need(ordo, "date de prescription", "date_prescription", "date_ordonnance", "date")
    c_val = col(ordo, "validite_jours", "validite", "duree_validite", contains=["valid"])
    c_typ = col(ordo, "type_ordonnance", "type")
    cols = ["ordonnance_id", c_od] + [
        c for c in (c_val, c_typ) if c and c not in ("ordonnance_id", c_od)
    ]
    O = (
        fac[["facture_id", C_ORD, C_DS]]
        .dropna(subset=[C_ORD])
        .merge(ordo[cols], left_on=C_ORD, right_on="ordonnance_id", how="left")
    )
    delay = (O[C_DS] - O[c_od]).dt.days
    if c_val and pd.api.types.is_numeric_dtype(O[c_val]):
        lim = O[c_val]
        resolved["6 validité"] = f"colonne {c_val}"
    else:
        is_opt = O[c_typ].astype(str).str.lower().str.contains("optiq") if c_typ else False
        lim = np.where(is_opt, 30, 10)
        resolved["6 validité"] = f"règle 10 j / 30 j optique (type={c_typ})"
    resolved["6 date prescription"] = c_od
    resolved["6 soins avant prescription"] = int((delay < 0).sum())
    return by_fact(delay > lim, O["facture_id"])


def s_ferme():
    c_j = need(pre, "jours d'ouverture", "jours_ouverture")
    resolved["7 exemples jours_ouverture"] = list(pre[c_j].astype(str).value_counts().head(5).index)

    def parse(v):
        s = str(v)
        m = re.fullmatch(r"\s*([1-7])\s*-\s*([1-7])\s*", s)
        if m:
            return set(range(int(m[1]), int(m[2]) + 1))
        return {int(x) for x in re.findall(r"[1-7]", s)}

    opens = pre.drop_duplicates("prestataire_id").set_index("prestataire_id")[c_j].map(parse)
    d = fac[["facture_id", C_PRE, C_DS]]
    od = d[C_PRE].map(opens)
    w = d[C_DS].dt.dayofweek + 1
    flag = [
        isinstance(s, set) and len(s) > 0 and pd.notna(k) and int(k) not in s for s, k in zip(od, w)
    ]
    return pd.Series(flag, index=d["facture_id"].values)


def s_doublon(keep):
    return pd.Series(
        fac.duplicated(subset=[C_ASS, C_PRE, C_MT, C_DS], keep=keep).values,
        index=fac["facture_id"].values,
    )


def s_fraction():
    g = fac.groupby([C_ASS, C_PRE, C_DS])["facture_id"].transform("size")
    return pd.Series((g >= 2).values, index=fac["facture_id"].values)


def s_surconso():
    d = fac.dropna(subset=[C_ORD])
    n = d.groupby(C_ORD)[C_PRE].transform("nunique")
    return pd.Series((n > 1).values, index=d["facture_id"].values)


def s_fictif_sans_ord():
    sans = fac[C_ORD].isna()
    c_req = col(act, "ordonnance_requise", contains=["ordonnance"])
    if c_req is None:
        resolved["11a"] = "pas de colonne ordonnance_requise : toute facture sans ordonnance"
        return pd.Series(sans.values, index=fac["facture_id"].values)
    req = act.loc[
        act[c_req].astype(str).str.lower().isin(["1", "1.0", "true", "oui", "o", "vrai", "yes"]),
        A_CODE,
    ]
    ids = lf.loc[(lf[C_LTYPE] == "ACTE") & lf[C_LCODE].isin(req), "facture_id"].unique()
    resolved["11a"] = f"sans ordonnance + acte exigeant une ordonnance ({c_req})"
    return pd.Series((sans & fac["facture_id"].isin(ids)).values, index=fac["facture_id"].values)


def s_fictif_autre():
    m = (
        fac[["facture_id", C_ASS, C_ORD]]
        .dropna(subset=[C_ORD])
        .merge(
            ordo[["ordonnance_id", "assure_id"]].rename(columns={"assure_id": "_ass_o"}),
            left_on=C_ORD,
            right_on="ordonnance_id",
            how="left",
        )
    )
    return pd.Series(
        (m["_ass_o"].notna() & (m["_ass_o"] != m[C_ASS])).values, index=m["facture_id"].values
    )


def s_profil():
    c_sx = need(ass, "sexe assuré", "sexe")
    c_by = need(ass, "année de naissance", "annee_naissance", contains=["naiss"])
    a_sx = col(act, "sexe_requis", contains=["sexe"])
    a_min = col(act, "age_min", contains=["age", "min"])
    a_max = col(act, "age_max", contains=["age", "max"])
    if not any((a_sx, a_min, a_max)):
        raise KeyError(
            f"Aucune restriction sexe/âge dans referentiel_actes. Colonnes : {list(act.columns)}"
        )
    ren = {A_CODE: "_acode"}
    if a_sx:
        ren[a_sx] = "_sx_req"
    if a_min:
        ren[a_min] = "_amin"
    if a_max:
        ren[a_max] = "_amax"
    a = act[list(ren)].rename(columns=ren).drop_duplicates("_acode")
    s_ = ass[["assure_id", c_sx, c_by]].rename(columns={c_sx: "_sx", c_by: "_by"})
    L = (
        lf[lf[C_LTYPE] == "ACTE"]
        .merge(a, left_on=C_LCODE, right_on="_acode", how="left")
        .merge(s_, left_on=C_ASS, right_on="assure_id", how="left")
    )
    resolved["12 modalités sexe assuré"] = L["_sx"].value_counts().head(4).to_dict()
    if "_sx_req" in L:
        resolved["12 modalités sexe requis"] = (
            L["_sx_req"].astype(str).value_counts().head(5).to_dict()
        )
    age = L[C_DS].dt.year - pd.to_numeric(L["_by"], errors="coerce")
    flag = pd.Series(False, index=L.index)
    if "_sx_req" in L:
        rq = L["_sx_req"].astype(str).str.strip().str.upper().str[:1]
        sx = L["_sx"].astype(str).str.strip().str.upper().str[:1]
        flag |= rq.isin(["M", "F"]) & sx.isin(["M", "F"]) & (rq != sx)
    if "_amin" in L:
        flag |= (age + 1) < pd.to_numeric(L["_amin"], errors="coerce")
    if "_amax" in L:
        flag |= (age - 1) > pd.to_numeric(L["_amax"], errors="coerce")
    return by_fact(flag, L["facture_id"])


def s_radie(mode):
    c_fin = need(pre, "fin de convention", "date_fin_convention", contains=["fin", "conv"])
    c_st = col(pre, "statut")
    ren = {c_fin: "_fin"}
    if c_st:
        ren[c_st] = "_st"
    P = pre[["prestataire_id"] + list(ren)].rename(columns=ren)
    d = fac[["facture_id", C_PRE, C_DS]].merge(
        P, left_on=C_PRE, right_on="prestataire_id", how="left"
    )
    if mode == "fin":
        flag = d["_fin"].notna() & (d[C_DS] > d["_fin"])
    else:
        if "_st" not in d:
            raise KeyError("colonne statut absente de prestataires.csv")
        flag = d["_st"].astype(str).str.upper().eq("RADIE")
    return pd.Series(flag.values, index=d["facture_id"].values)


def s_hors_couv():
    c_ad = need(ass, "adhésion", "date_adhesion", contains=["adh"])
    c_rad = need(ass, "radiation", "date_radiation", contains=["radi"])
    c_eff = need(con, "date d'effet", "date_effet", contains=["effet"])
    c_cf = need(con, "fin de contrat", "date_fin", contains=["fin"])
    resolved["14 colonnes"] = f"{c_ad}, {c_rad}, {c_eff}, {c_cf}"
    A = ass[["assure_id", c_ad, c_rad]].rename(columns={c_ad: "_ad", c_rad: "_rad"})
    Cn = con[["contrat_id", c_eff, c_cf]].rename(columns={c_eff: "_eff", c_cf: "_cfin"})
    d = (
        fac[["facture_id", C_ASS, C_CTR, C_DS]]
        .merge(A, left_on=C_ASS, right_on="assure_id", how="left")
        .merge(Cn, left_on=C_CTR, right_on="contrat_id", how="left")
    )
    ds = d[C_DS]
    flag = (
        (ds < d["_ad"])
        | (d["_rad"].notna() & (ds > d["_rad"]))
        | (ds < d["_eff"])
        | (d["_cfin"].notna() & (ds > d["_cfin"]))
    )
    return pd.Series(flag.values, index=d["facture_id"].values)


run_signal("01 prix acte > tarif max référentiel", "SURFACTURATION", s_surf_acte)
run_signal("01b prix médicament > prix réf.", "SURFACTURATION", s_surf_med)
run_signal("01c prix > tarif grille prestataire", "SURFACTURATION", lambda: s_surf_grille(1.0))
run_signal("01d prix > 1,10 x tarif grille", "SURFACTURATION", lambda: s_surf_grille(1.10))
run_signal("02 quantité facturée > prescrite", "QUANTITE_EXCESSIVE", s_qte)
run_signal("04 ligne absente de l'ordonnance", "LIGNES_NON_PRESCRITES", s_non_prescrites)
run_signal("05 médicament non prescrit cher", "SUBSTITUTION_PRODUIT", s_substitution)
run_signal("06 soin après validité ordonnance", "ORDONNANCE_EXPIREE", s_expiree)
run_signal("07 soin un jour de fermeture", "SOINS_JOUR_FERME", s_ferme)
run_signal("08a doublon (copie suivante)", "DOUBLON", lambda: s_doublon("first"))
run_signal("08b doublon (tous membres)", "DOUBLON", lambda: s_doublon(False))
run_signal("09 >= 2 factures même jour/prestataire/assuré", "FRACTIONNEMENT", s_fraction)
run_signal("10 ordonnance chez >= 2 prestataires", "SURCONSOMMATION", s_surconso)
run_signal("11a sans ordonnance (acte exigeant)", "ACTE_FICTIF", s_fictif_sans_ord)
run_signal("11b ordonnance d'un autre assuré", "ACTE_FICTIF", s_fictif_autre)
run_signal("12 acte incompatible sexe/âge", "INCOHERENCE_PROFIL", s_profil)
run_signal("13a soin après fin de convention", "PRESTATAIRE_RADIE", lambda: s_radie("fin"))
run_signal("13b prestataire au statut RADIE", "PRESTATAIRE_RADIE", lambda: s_radie("statut"))
run_signal("14 hors période de couverture", "HORS_COUVERTURE", s_hors_couv)
print(
    "UPCODING : aucun signal fiable sans heure ni spécialité de l'acte ; NON ÉVALUÉ dans cette EDA (voir section 8)."
)

valid = y.notna()
prev = float(y[valid].mean())
rows = []
for name, (expected, s, err) in results.items():
    if err:
        rows.append(
            {"signal": name, "type_attendu": expected, "statut": "NON ÉVALUÉ", "detail": err}
        )
        continue
    fl = s & valid
    n = int(fl.sum())
    tp = int((fl & (y == 1)).sum())
    mt = (typ == expected) & valid
    prec = tp / n if n else np.nan
    rows.append(
        {
            "signal": name,
            "type_attendu": expected,
            "statut": "évalué",
            "n_flag": n,
            "pct_factures": round(n / int(valid.sum()) * 100, 2),
            "precision": round(prec, 3) if n else np.nan,
            "lift": round(prec / prev, 1) if n else np.nan,
            "rappel_global": round(tp / N_F, 3),
            "rappel_type_attendu": round(int((fl & mt).sum()) / int(mt.sum()), 3)
            if int(mt.sum())
            else np.nan,
        }
    )
sig_df = pd.DataFrame(rows)
print("\nColonnes résolues / constats :")
for k_, v_ in resolved.items():
    print(f"  {k_}: {v_}")
print(f"\nTaux de fraude global (référence) : {prev:.2%}")
print(sig_df.drop(columns=["detail"], errors="ignore").to_string(index=False))
if "detail" in sig_df:
    print("\nRaisons des signaux non évalués :")
    for _, r_ in sig_df[sig_df["statut"] == "NON ÉVALUÉ"].iterrows():
        print(f"  {r_['signal']} : {r_['detail']}")

UPCODING : aucun signal fiable sans heure ni spécialité de l'acte ; NON ÉVALUÉ dans cette EDA (voir section 8).

Colonnes résolues / constats :
  1 tarif max (réf. actes): tarif_max
  1b prix réf. (médicaments): prix_reference
  1c/1d tarif grille: tarif_pratique
  2 ordonnance_lignes code/qté: code / quantite
  5 seuil prix médicament (P75 réf.): 12750.0
  6 validité: colonne validite_jours
  6 date prescription: date_prescription
  6 soins avant prescription: 138
  7 exemples jours_ouverture: ['123456', '12345', '1234567']
  11a: sans ordonnance + acte exigeant une ordonnance (ordonnance_requise)
  12 modalités sexe assuré: {'F': 31405, 'M': 30389}
  12 modalités sexe requis: {'F': 914, 'M': 125}
  14 colonnes: date_adhesion, date_radiation, date_effet, date_fin

Taux de fraude global (référence) : 8.69%
                                       signal          type_attendu statut  n_flag  pct_factures  precision  lift  rappel_global  rappel_type_attendu
         01 prix acte > tarif ma

## 6. Déséquilibre de classes

In [ ]:
y1 = lab_ok["est_fraude"]
n1, n0 = int((y1 == 1).sum()), int((y1 == 0).sum())
ratio = n0 / n1
print(
    f"Classe 0 : {n0:,} ({n0 / N:.2%}) | classe 1 : {n1:,} ({n1 / N:.2%}) | ratio {ratio:.1f} : 1"
)
print(f"Exactitude d'un modèle qui prédit toujours « non fraude » : {n0 / N:.2%}")
print(f"PR-AUC d'un classifieur aléatoire (= prévalence) : {n1 / N:.4f}")
print(f"scale_pos_weight de départ (n0/n1) : {ratio:.1f}")
print(
    f"Taux mensuel de fraude : min {par_mois['taux_pct'].min()} % | max {par_mois['taux_pct'].max()} % sur {len(par_mois)} mois"
)
print(f"Types les plus rares : {tf.tail(3).to_dict()}")
print(
    f"Groupes pour un découpage groupé : {lab_ok['assure_id'].nunique():,} assurés, {lab_ok['prestataire_id'].nunique():,} prestataires"
)
print("\nImplications, à arbitrer en phase de modélisation :")
print("- Métriques : PR-AUC, rappel à précision fixée, F1 de la classe positive. Pas d'exactitude.")
print(
    "- Pondération (scale_pos_weight, class_weight) en premier choix : aucune donnée synthétique créée."
)
print(
    "- Sous-échantillonnage des négatifs possible ; SMOTE à manier avec prudence (lignes synthétiques, à appliquer sur le train seulement)."
)
print(
    "- Seuil de décision réglé sur la courbe précision-rappel : il alimente fraud_score_threshold (config)."
)
print(
    "- Découpage : temporel et/ou groupé par assuré/prestataire, pour éviter les fuites entre train et test."
)

Classe 0 : 77,246 (91.31%) | classe 1 : 7,347 (8.69%) | ratio 10.5 : 1
Exactitude d'un modèle qui prédit toujours « non fraude » : 91.31%
PR-AUC d'un classifieur aléatoire (= prévalence) : 0.0869
scale_pos_weight de départ (n0/n1) : 10.5
Taux mensuel de fraude : min 7.79 % | max 100.0 % sur 25 mois
Types les plus rares : {'SOINS_JOUR_FERME': 227, 'HORS_COUVERTURE': 195, 'PRESTATAIRE_RADIE': 147}
Groupes pour un découpage groupé : 4,375 assurés, 249 prestataires

Implications, à arbitrer en phase de modélisation :
- Métriques : PR-AUC, rappel à précision fixée, F1 de la classe positive. Pas d'exactitude.
- Pondération (scale_pos_weight, class_weight) en premier choix : aucune donnée synthétique créée.
- Sous-échantillonnage des négatifs possible ; SMOTE à manier avec prudence (lignes synthétiques, à appliquer sur le train seulement).
- Seuil de décision réglé sur la courbe précision-rappel : il alimente fraud_score_threshold (config).
- Découpage : temporel et/ou groupé par assuré/prest

## 7. Problèmes de qualité (mesurés)

Chaque ligne est calculée. La gravité est déduite du pourcentage mesuré : 0 = OK, moins de 0,5 % = Faible, moins de 5 % = Moyenne, au-delà = Haute. Les contraintes de modélisation (fuite, déséquilibre, PII) sont listées à part : ce ne sont pas des défauts de qualité.

In [ ]:
from contextlib import contextmanager

fac = dfs["factures.csv"]
checks = []


def add(cat, constat, n, base, note="", grav=None):
    n = int(n)
    pct = (n / base * 100) if base else 0.0
    g = grav or (
        "OK" if n == 0 else ("Faible" if pct < 0.5 else ("Moyenne" if pct < 5 else "Haute"))
    )
    checks.append(
        {
            "ID": len(checks) + 1,
            "Catégorie": cat,
            "Constat": constat,
            "Volume": n,
            "Base": base,
            "Pct": round(pct, 2),
            "Gravité": g,
            "Note": note,
        }
    )


@contextmanager
def safe(label):
    try:
        yield
    except Exception as e:
        checks.append(
            {
                "ID": len(checks) + 1,
                "Catégorie": "-",
                "Constat": label,
                "Volume": None,
                "Base": None,
                "Pct": None,
                "Gravité": "NON ÉVALUÉ",
                "Note": f"{type(e).__name__}: {str(e)[:110]}",
            }
        )


with safe("Dates non parsables"):
    tot = 0
    for name, dd in date_coerce.items():
        for c, n in dd.items():
            tot += n
            if n:
                add("Dates", f"{name}.{c} non parsable (AAAA-MM-JJ)", n, len(dfs[name]))
    if tot == 0:
        add("Dates", "Dates non parsables (tous fichiers)", 0, 1)

c_df = col(fac, "date_facture", "date_emission")
c_dr = col(fac, "date_reception")
with safe("date_soin > date_facture"):
    add(
        "Cohérence dates",
        f"date_soin > {c_df}",
        (fac["date_soin"] > fac[c_df]).sum(),
        fac[[c_df, "date_soin"]].dropna().shape[0],
    )
with safe("date_facture > date_reception"):
    add(
        "Cohérence dates",
        f"{c_df} > {c_dr}",
        (fac[c_df] > fac[c_dr]).sum(),
        fac[[c_df, c_dr]].dropna().shape[0],
    )

C_MT_ = need(fac, "montant total", "montant_total", contains=["montant", "total"])
lig = dfs["lignes_facture.csv"]
c_lmt = col(lig, "montant", "montant_ligne", "montant_total", contains=["montant"])
c_lpu = col(lig, "prix_unitaire", contains=["prix"])
c_lq = col(lig, "quantite", contains=["quant"])
with safe("Montants <= 0 (factures)"):
    add("Montants", f"{C_MT_} <= 0 (factures)", (pd.to_numeric(fac[C_MT_]) <= 0).sum(), len(fac))
with safe("Montants <= 0 (lignes)"):
    add(
        "Montants",
        "lignes avec montant, prix ou quantité <= 0",
        ((lig[[c_lmt, c_lpu, c_lq]] <= 0).any(axis=1)).sum(),
        len(lig),
    )
with safe("Total facture vs somme des lignes"):
    s_ = lig.groupby("facture_id")[c_lmt].sum()
    m_ = fac.set_index("facture_id")[C_MT_]
    com = m_.index.intersection(s_.index)
    add("Montants", "montant total != somme des lignes", (m_[com] != s_[com]).sum(), len(com))
with safe("Montant ligne vs quantité x prix"):
    add(
        "Montants",
        "montant ligne != quantité x prix unitaire",
        (lig[c_lmt] != lig[c_lq] * lig[c_lpu]).sum(),
        len(lig),
    )
with safe("Demande + part assuré vs total"):
    c_dem = need(fac, "montant demandé", "montant_demande", contains=["demand"])
    c_part = need(fac, "part assuré", "part_assure", contains=["part"])
    add(
        "Montants",
        f"{c_dem} + {c_part} != {C_MT_}",
        (fac[c_dem] + fac[c_part] != fac[C_MT_]).sum(),
        len(fac),
    )
with safe("Codes de lignes inconnus"):
    add(
        "Référentiels",
        "lignes ACTE au code absent du référentiel",
        orph_codes_act,
        int((lig[c_ltype0] == "ACTE").sum()),
    )
    add(
        "Référentiels",
        "lignes MEDICAMENT au code absent du référentiel",
        orph_codes_med,
        int((lig[c_ltype0] == "MEDICAMENT").sum()),
    )
with safe("Factures sans ligne"):
    add(
        "Intégrité",
        "factures sans ligne",
        len(set(fac["facture_id"]) - set(lig["facture_id"])),
        len(fac),
    )
with safe("Ordonnances sans ligne"):
    o_ = dfs["ordonnances.csv"]
    add(
        "Intégrité",
        "ordonnances sans ligne",
        len(set(o_["ordonnance_id"]) - set(dfs["ordonnance_lignes.csv"]["ordonnance_id"])),
        len(o_),
    )
with safe("Statuts de factures"):
    add(
        "Valeurs",
        "statut hors PAYEE / REJETEE / EN_COURS",
        (~fac["statut"].isin(["PAYEE", "REJETEE", "EN_COURS"])).sum(),
        len(fac),
        str(fac["statut"].value_counts(dropna=False).to_dict()),
    )
with safe("Statuts de contrats"):
    c_ = dfs["contrats.csv"]
    cs = col(c_, "statut")
    add(
        "Valeurs",
        "modalités de statut des contrats (info)",
        0,
        len(c_),
        str(c_[cs].value_counts(dropna=False).to_dict()),
        grav="Info",
    )
with safe("Contrats : fin avant effet"):
    c_ = dfs["contrats.csv"]
    ce, cf = (
        need(c_, "date d'effet", "date_effet", contains=["effet"]),
        need(c_, "fin de contrat", "date_fin", contains=["fin"]),
    )
    add("Cohérence dates", "contrats avec fin < effet", (c_[cf] < c_[ce]).sum(), len(c_))
with safe("Assurés : radiation avant adhésion"):
    a_ = dfs["assures.csv"]
    ca, cr = (
        need(a_, "adhésion", "date_adhesion", contains=["adh"]),
        need(a_, "radiation", "date_radiation", contains=["radi"]),
    )
    add("Cohérence dates", "assurés avec radiation < adhésion", (a_[cr] < a_[ca]).sum(), len(a_))
with safe("Prestataires radiés sans date de fin"):
    p_ = dfs["prestataires.csv"]
    cst = need(p_, "statut prestataire", "statut")
    cfn = need(p_, "fin de convention", "date_fin_convention", contains=["fin", "conv"])
    add(
        "Cohérence",
        "prestataires RADIE sans date_fin_convention",
        (p_[cst].astype(str).str.upper().eq("RADIE") & p_[cfn].isna()).sum(),
        len(p_),
    )
with safe("Ordonnance postérieure au soin"):
    O = (
        fac[["facture_id", "ordonnance_id", "date_soin"]]
        .dropna(subset=["ordonnance_id"])
        .merge(
            dfs["ordonnances.csv"][
                [
                    "ordonnance_id",
                    need(
                        dfs["ordonnances.csv"],
                        "date prescription",
                        "date_prescription",
                        "date_ordonnance",
                        "date",
                    ),
                ]
            ].rename(columns=lambda c: "_dp" if c.startswith("date") else c),
            on="ordonnance_id",
            how="left",
        )
    )
    add(
        "Cohérence dates",
        "date de prescription > date de soin",
        (O["_dp"] > O["date_soin"]).sum(),
        len(O),
    )
with safe("Colonnes constantes ou vides"):
    const = []
    for name, dd in dfs.items():
        if name != PII:
            const += [f"{name}.{c}" for c in dd.columns if dd[c].nunique(dropna=False) <= 1]
    add(
        "Schéma",
        "colonnes constantes ou entièrement vides",
        len(const),
        sum(dd.shape[1] for n_, dd in dfs.items() if n_ != PII),
        ", ".join(const[:8]),
    )
with safe("Manquants non structurels"):
    structurels = {
        "motif_rejet",
        "date_traitement",
        "montant_pris_en_charge",
        "ordonnance_id",
        "date_radiation",
        "date_fin_convention",
    }
    autres = sorted(
        {f"{r['fichier']}.{r['colonne']}" for r in miss_rows if r["colonne"] not in structurels}
    )
    add(
        "Valeurs",
        "colonnes avec manquants hors nulls attendus",
        len(autres),
        1,
        ", ".join(autres[:8]),
        grav="Info" if not autres else "Moyenne",
    )
with safe("Factures sans ordonnance"):
    add(
        "Valeurs",
        "factures sans ordonnance_id (structurel, voir ACTE_FICTIF)",
        fac["ordonnance_id"].isna().sum(),
        len(fac),
        grav="Info",
    )

df_q = pd.DataFrame(checks)
print("=== 7.1 PROBLÈMES DE QUALITÉ MESURÉS ===")
print(df_q.to_string(index=False))

print("\n=== 7.2 CONTRAINTES DE MODÉLISATION (hors qualité) ===")
fmax = leak_df.sort_values("force", ascending=False).iloc[0] if len(leak_df) else None
print(
    pd.DataFrame(
        [
            {
                "Contrainte": "Fuite de données",
                "Mesure": f"force max {fmax['force']} ({fmax['variable']})"
                if fmax is not None
                else "n/a",
                "Action": f"exclure {EXCLUDED_FEATURES}",
            },
            {
                "Contrainte": "Déséquilibre de classes",
                "Mesure": f"{ratio:.1f} : 1 ({n1 / N:.2%} de fraudes)",
                "Action": "pondération, PR-AUC, seuil réglé",
            },
            {
                "Contrainte": "Données personnelles",
                "Mesure": f"{pii_info['lignes']:,} lignes dans identites_pii.csv",
                "Action": "chiffrement pgcrypto, jamais dans les features",
            },
        ]
    ).to_string(index=False)
)

=== 7.1 PROBLÈMES DE QUALITÉ MESURÉS ===
 ID       Catégorie                                                    Constat  Volume     Base     Pct    Gravité                                                                                                                                                                                                                                                       Note
  1           Dates                        Dates non parsables (tous fichiers)     0.0      1.0    0.00         OK                                                                                                                                                                                                                                                           
  2 Cohérence dates                                   date_soin > date_facture     0.0  84593.0    0.00         OK                                                                                                                 

## 8. Synthèse générée à partir des résultats

La cellule ci-dessous reprend les chiffres calculés plus haut. Rien n'est écrit en dur.

In [ ]:
print("=== SYNTHÈSE ===")
print(f"- Factures étiquetées : {N:,} ; fraudes : {N_F:,} ({TAUX:.2%}) ; ratio {ratio:.1f} : 1.")
print(
    f"- Types de fraude : {len(present)} présents sur 14 attendus. Manquants : {missing_types or 'aucun'}. Inattendus : {extra_types or 'aucun'}."
)
print(f"- Orphelins de clés étrangères (total) : {orph_total:,}.")
if len(leak_df):
    top = leak_df.sort_values("force", ascending=False).head(3)
    print(
        "- Fuite : variables les plus liées à la fraude : "
        + " ; ".join(f"{r['variable']} (force {r['force']})" for _, r in top.iterrows())
    )
ev = sig_df[sig_df["statut"] == "évalué"]
if len(ev):
    best = ev.sort_values("lift", ascending=False).head(5)
    print(
        "- Signaux au meilleur lift : "
        + " ; ".join(
            f"{r['signal']} (lift {r['lift']}, précision {r['precision']})"
            for _, r in best.iterrows()
        )
    )
    faibles = ev[(ev["rappel_type_attendu"].fillna(0) < 0.5)]["signal"].tolist()
    print(f"- Signaux captant moins de 50 % de leur type : {faibles or 'aucun'}")
nonev = sig_df[sig_df["statut"] == "NON ÉVALUÉ"]["signal"].tolist()
print(f"- Signaux non évalués : {nonev or 'aucun'} + UPCODING (non évalué par construction).")
haute = df_q[df_q["Gravité"].isin(["Haute"])]["Constat"].tolist()
moy = df_q[df_q["Gravité"].isin(["Moyenne"])]["Constat"].tolist()
nev = df_q[df_q["Gravité"] == "NON ÉVALUÉ"]["Constat"].tolist()
print(f"- Qualité, gravité Haute : {haute or 'aucune'}")
print(f"- Qualité, gravité Moyenne : {moy or 'aucune'}")
print(f"- Contrôles qualité non évalués : {nev or 'aucun'}")
print(
    "- À relire par l'équipe : tout écart entre ces chiffres et le dictionnaire de données doit être signalé, pas corrigé ici."
)

=== SYNTHÈSE ===
- Factures étiquetées : 84,593 ; fraudes : 7,347 (8.69%) ; ratio 10.5 : 1.
- Types de fraude : 14 présents sur 14 attendus. Manquants : aucun. Inattendus : aucun.
- Orphelins de clés étrangères (total) : 0.
- Fuite : variables les plus liées à la fraude : montant_total (référence : variable légitime) (force 0.726) ; pris_en_charge / montant_total (force 0.673) ; statut (modalités) (force 0.594)
- Signaux au meilleur lift : 05 médicament non prescrit cher (lift 11.5, précision 1.0) ; 04 ligne absente de l'ordonnance (lift 11.5, précision 1.0) ; 02 quantité facturée > prescrite (lift 11.5, précision 1.0) ; 12 acte incompatible sexe/âge (lift 11.5, précision 1.0) ; 13a soin après fin de convention (lift 11.5, précision 1.0)
- Signaux captant moins de 50 % de leur type : ['01b prix médicament > prix réf.', '11a sans ordonnance (acte exigeant)', "11b ordonnance d'un autre assuré"]
- Signaux non évalués : aucun + UPCODING (non évalué par construction).
- Qualité, gravité Hau